# Create and read a person

`POST /admin/people` adds a directory row. The connector stores canonical
JSON at `raw/hr/person/{id}.json`: sorted keys, and lists sorted so the hash
does not flap. The snapshot is the public record. Names in `hr_demo.FORBIDDEN`
are not on it.


In [1]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition and create


In [2]:
PERSON_ID = "emp-900001"
KEY = f"raw/hr/person/{PERSON_ID}.json"
PAYLOAD = {
    "person_id": PERSON_ID,
    "email": "emp-900001@example.com",
    "full_name": "Demo Person",
    "team": "people",
    "role": "People Partner",
    "region": "DE",
    "manager_id": None,
    "native_acl": ["hr_employee"],
}
hr_demo.delete_if_present("person", PERSON_ID)
created = hr_demo.admin("POST", "/admin/people", PAYLOAD)
print(created["person_id"], "version", created["version"], created["native_acl"])


emp-900001 version 1 ['hr_employee']


## Snapshot


In [3]:
import json

def ready() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    if not all(meta.get(name) for name in hr_demo.META_KEYS):
        return False
    if meta.get(hr_demo.META_VERSION) != str(created["version"]):
        return False
    raw = hr_demo.object_body(KEY)
    record = json.loads(raw)
    leaked = set(record) & hr_demo.FORBIDDEN
    return raw == hr_demo.canonical(record) and not leaked

hr_demo.wait_until(ready, hr_demo.WRITE_WAIT, "person snapshot")
record = json.loads(hr_demo.object_body(KEY))
assert record == hr_demo.admin("GET", f"/people/{PERSON_ID}")
print(sorted(record))
print(hr_demo.snapshot(KEY)["Metadata"])


TimeoutError: person snapshot did not finish: False

## Negative cases


In [ ]:
duplicate = hr_demo.expect_status("POST", "/admin/people", 409, PAYLOAD)
print("duplicate", duplicate.status_code)
bad = {**PAYLOAD, "person_id": "emp-1"}
rejected = hr_demo.expect_status("POST", "/admin/people", 422, bad)
print("bad id", rejected.status_code)


## Cleanup


In [ ]:
hr_demo.delete_if_present("person", PERSON_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)
